In [ ]:
# 1. Подключаем Google Диск
from google.colab import drive
import os
drive.mount('/content/drive')

# 2. Создаем чистую папку в локальном хранилище Colab (это SSD, там обучение пойдет быстро)
LOCAL_DATA_DIR = "/content/extracted_dataset"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# 3. Распаковываем ZIP-архив с Диска в созданную локальную папку
# ЗАМЕНЯЙТЕ 'dataset.zip' на точное имя вашего архива на Диске!
!unzip -q "/content/drive/MyDrive/image_similarity_project/dl-lab-5-metric-learning.zip" -d {LOCAL_DATA_DIR}

print("Распаковка успешно завершена!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/extracted_dataset/submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: Распаковка успешно завершена!


In [ ]:
# ============================================================
# 1. УСТАНОВКА И ИМПОРТ БИБЛИОТЕК
# ============================================================
!pip install -q transformers datasets accelerate tqdm pandas

import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel
from tqdm import tqdm
from google.colab import drive

# Автоматическое сохранение на Google Диск во избежание потери прогресса
drive.mount('/content/drive')

DRIVE_BASE_DIR = Path("/content/drive/MyDrive/image_similarity_project")
SUBMISSIONS_DIR = DRIVE_BASE_DIR / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# НАСТРОЙКА ПУТЕЙ (на основе вашей структуры папок)
# ------------------------------------------------------------
BASE_EXTRACTED = Path("/content/extracted_dataset/")

INPUT_SUBMISSION_PATH = BASE_EXTRACTED / "submission.csv"
OUTPUT_SUBMISSION_PATH = SUBMISSIONS_DIR / "final_pure_vision_submission.csv"
IMAGE_ROOT_DIR = BASE_EXTRACTED.resolve()

MODEL_NAME = "facebook/dinov2-vit-large-patch14"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
BATCH_SIZE = 32  # Оптимально для премиум-GPU (V100/A100) в Colab Pro
SEED = 42

# Строгая фиксация сидов для воспроизводимости
torch.manual_seed(SEED)
np.random.seed(SEED)

# ============================================================
# 2. ИНДЕКСАЦИЯ ПУТЕЙ (Исключительно для локализации файлов)
# ============================================================
def build_file_registry(root: Path) -> dict:
    """
    Рекурсивно обходит директории и сопоставляет имя файла с его путем.
    Если файл с таким именем уже был найден в другой подпапке,
    мы его пропускаем, чтобы избежать дублирования путей.
    """
    registry = {}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            # Вместо raise ValueError просто пропускаем дубликаты путей
            if path.name not in registry:
                registry[path.name] = path
    return registry


print("Индексирую файлы изображений...")
file_registry = build_file_registry(IMAGE_ROOT_DIR)
print(f"Индексация завершена. Всего файлов в кэше: {len(file_registry)}")

# ============================================================
# 3. КЛАСС DATASET (РАБОТАЕТ СТРОГО С ПИКСЕЛЯМИ)
# ============================================================
class StrictVisionDataset(Dataset):
    def __init__(self, df, registry, processor):
        self.df = df.reset_index(drop=True)
        self.registry = registry
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        f1, f2 = str(row['file_1']), str(row['file_2'])

        # Чтение картинок с диска
        img1 = Image.open(self.registry[f1]).convert("RGB")
        img2 = Image.open(self.registry[f2]).convert("RGB")

        # Официальный препроцессинг DINOv3 (изменение размера, стандартизация)
        inputs1 = self.processor(images=img1, return_tensors="pt")
        inputs2 = self.processor(images=img2, return_tensors="pt")

        return {
            "pix1": inputs1["pixel_values"].squeeze(0),
            "pix2": inputs2["pixel_values"].squeeze(0)
        }

# Инициализация модели и процессора на GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()  # Строгий режим инференса

# ============================================================
# 4. ЧЕСТНЫЙ ИНФЕРЕНС (1 МОДЕЛЬ, 1 ПРОХОД ПО ИЗОБРАЖЕНИЮ)
# ============================================================
def run_pure_vision_inference(model, dataloader, device):
    all_similarities = []

    # Полное отключение градиентов для максимальной скорости и экономии VRAM
    with torch.inference_mode():
        for batch in tqdm(dataloader, desc="Вычисление визуального сходства"):
            pix1 = batch["pix1"].to(device)
            pix2 = batch["pix2"].to(device)

            # Извлечение признаков за один прямой проход (Single forward pass)
            outputs1 = model(pixel_values=pix1)
            outputs2 = model(pixel_values=pix2)

            emb1 = outputs1.pooler_output
            emb2 = outputs2.pooler_output

            # L2-нормализация векторов (согласно базовому решению)
            emb1 = torch.nn.functional.normalize(emb1, p=2, dim=1)
            emb2 = torch.nn.functional.normalize(emb2, p=2, dim=1)

            # Попарное косинусное сходство через скалярное произведение всего батча на GPU
            sim = torch.sum(emb1 * emb2, dim=1)

            all_similarities.extend(sim.cpu().numpy())

    return np.array(all_similarities)

# ============================================================
# 5. СБОРКА И ЗАПУСК ПАЙПЛАЙНА
# ============================================================
print("\n--- Загрузка таблицы submission.csv ---")
submission_df = pd.read_csv(INPUT_SUBMISSION_PATH)

test_dataset = StrictVisionDataset(submission_df, file_registry, processor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("\n--- Вычисление сходства (Чистый визуал без TTA/Ансамблей) ---")
final_predictions = run_pure_vision_inference(model, test_loader, device)

# Перезаписываем колонку similarity новыми точными скорами от SOTA-модели
submission_df["similarity"] = final_predictions

# Сохранение на Google Диск
submission_df.to_csv(OUTPUT_SUBMISSION_PATH, index=False, encoding="utf-8-sig")

print("\nРабота скрипта успешно завершена!")
print(f"Итоговый честный submission сохранен на ваш Диск: {OUTPUT_SUBMISSION_PATH}")
print("\nСтатистика распределения новых предсказаний:")
print(submission_df["similarity"].describe())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Индексирую файлы изображений...
Индексация завершена. Всего файлов в кэше: 21110
Используется устройство: cuda


OSError: facebook/dinov2-vit-large-patch14 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`